# 15. Graph Machine Learning

Convert a small TopologicPy graph into PyTorch Geometric data and train one graph convolution layer for a few steps. This is a mechanics demonstration, not evidence of useful predictive performance.

**Goal:** Complete one small workflow. Run the cells from top to bottom in a fresh kernel.

**Extra dependencies:** CPU `torch==2.14.1` and `torch-geometric==2.8.1`. Package installation is separate from execution.

All examples use metres unless stated otherwise.


In [ ]:
import os
os.environ["TOPOLOGICPY_CORE_BACKEND"] = "pythonocc"

import topologicpy
from topologicpy.Core import Core

assert topologicpy.__version__ == "0.9.86", "Use TopologicPy 0.9.86 for these tutorials."
assert type(Core.Backend()).__name__ == "PythonOCCBackend", "Restart with the pythonocc backend."
print("TopologicPy", topologicpy.__version__, "—", type(Core.Backend()).__name__)


## Build and convert a graph

The four-space graph comes from TopologicPy. Graph edges are undirected, so PyG receives an index pair for each direction. Features describe X and Y coordinates; labels distinguish the hall from the other spaces.


In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.TGraph import TGraph

entrance = Vertex.ByCoordinates(0, 0, 0)
hall = Vertex.ByCoordinates(2, 0, 0)
studio = Vertex.ByCoordinates(4, 0, 0)
office = Vertex.ByCoordinates(2, 2, 0)
vertices = [entrance, hall, studio, office]
edges = [{"src": 0, "dst": 1},
         {"src": 1, "dst": 2},
         {"src": 1, "dst": 3}]
graph = TGraph.ByVerticesEdges(vertices, edges, directed=False)
assert isinstance(graph, TGraph)
assert len(TGraph.Vertices(graph)) == 4
assert len(TGraph.Edges(graph)) == 3


import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

torch.manual_seed(7)
torch.set_num_threads(1)
mesh = TGraph.MeshData(graph)
features = []
for coordinates in mesh["vertices"]:
    features.append(coordinates[:2])
edge_pairs = []
for start, end in mesh["edges"]:
    edge_pairs.append([start, end])
    edge_pairs.append([end, start])
data = Data(x=torch.tensor(features, dtype=torch.float32),
            edge_index=torch.tensor(edge_pairs, dtype=torch.long).t().contiguous(),
            y=torch.tensor([0, 1, 0, 0], dtype=torch.long))
assert data.num_nodes == len(TGraph.Vertices(graph))
assert data.num_edges == 2 * len(TGraph.Edges(graph))
print(data)


## Run a tiny training loop

A graph convolution mixes information from neighbours. The model returns two scores per node; cross entropy compares those scores with the labels. We use all four nodes for training, so the result below is training behaviour only. A real experiment needs separate training, validation and test data.


In [ ]:
layer = GCNConv(2, 2)
optimiser = torch.optim.Adam(layer.parameters(), lr=0.05)
initial_loss = torch.nn.functional.cross_entropy(layer(data.x, data.edge_index), data.y).item()
for epoch in range(30):
    optimiser.zero_grad()
    scores = layer(data.x, data.edge_index)
    loss = torch.nn.functional.cross_entropy(scores, data.y)
    loss.backward()
    optimiser.step()
print("Initial loss:", round(initial_loss, 4), "Final loss:", round(loss.item(), 4))
with torch.no_grad():
    predictions = layer(data.x, data.edge_index).argmax(dim=1)
print("Training predictions:", predictions.tolist())
assert torch.isfinite(loss).item()
assert loss.item() < initial_loss
assert len(predictions) == 4


## Try it yourself

Change the node features and observe the training loss. For a real task, first define an independent test set and a baseline; then explore `topologicpy.PyG` for larger dataset workflows.
